### Import Dependencies

In [5]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.agents.run_config import RunConfig
from google.genai import types
from utils.tools_2 import check_warehouse_availability, reserve_warehouse_items

import os
from dotenv import load_dotenv

load_dotenv('../../.env')



True

### ADK Agent
- when using models other than Gemini, you need to use a model router like LiteLlm
- the agent tools needed to be defined as function tools, because the original ones under tools.py are Langchain tools (see @tool decorator) which is not supported by ADK

In [ ]:
model = LiteLlm(
    model="openai/gpt-5.4-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [7]:
warehouse_agent = Agent(
    name="warehouse_manager_agent", #mandatory
    model=model,
    tools=[
        check_warehouse_availability,
        reserve_warehouse_items
    ],
    description="""
    The warehouse manager agent is responsible for checking the availability of items in the warehouse and reserving them.
    """,
    instruction="""
    You are a part of the shopping assistant that can manage available inventory in the warehouses.

    ## Instructions

    - As the final answer you should return an answer to the users query in a form of actions performed.
    - You must always check the availability of the items in the warehouses before reserving them.
    - Only reserve items in warehouses if entire order can be reserved or the user has confirmed that they want a partial reservation.
    - If you cannot reserve any items, return an answer that the order cannot be reserved.
    - If you can reserve some items, return an answer that the order can be partially reserved and include the details.
    - If only partial quantity can be reserved in some warehouses, try to combine the required quantity from different warehouses.
    - Try to reserve items from the closest warehouse to the user first if users location is provided.
    - As the final answer you should return an answer in a form of actions performed.
    """
)

### Define the ADK session

In [8]:
session_service = InMemorySessionService()

In [9]:
await session_service.create_session(
    app_name='warehouse_app',
    user_id='user_1',
    session_id='session_1'
)

Session(id='session_1', app_name='warehouse_app', user_id='user_1', state={}, events=[], last_update_time=1786911596.4567966)

### Define the ADK runner

In [ ]:
runner = Runner(
    session_service=session_service,
    agent=warehouse_agent,
    app_name='warehouse_app'
)

### Executing the runner
- the runner is the agent execution engine

In [11]:
message = types.Content(
    role="user",
    parts=[
        types.Part(
            text="What's the availability of B09X1LDMH6 in all warehouses?"
        )
    ]
)

In [13]:
result = runner.run(
    user_id='user_1',
    session_id='session_1',
    new_message=message,
    run_config=RunConfig(
        max_llm_calls=5
    )
)

In [16]:
for event in result:
    print(event)

/home/pcrespo/Documents/estudos/github_repos/e2e_ai_engineering_bootcamp/.venv/lib/python3.11/site-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


model_version='gpt-5.4-mini' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'items': [
            {<... 2 items at Max depth ...>},
          ]
        },
        id='call_2JPTCKkI0y5vORFdScFZ8zS4',
        name='check_warehouse_availability'
      )
    ),
  ],
  role='model'
) grounding_metadata=None partial=False turn_complete=None turn_complete_reason=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  cached_content_token_count=0,
  candidates_token_count=37,
  prompt_token_count=574,
  total_token_count=611
) live_session_resumption_update=None live_session_id=None go_away=None voice_activity=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None environment_id=None invocation_id='e-b47e273a-30a9-44ed-b49b-3d

### Implementing an agent run wrapper

In [17]:
async def run_warehouse_agent(query: str, session_id: str, user_id: str, session_service: InMemorySessionService) -> str:
    existing_session = await session_service.get_session(
        session_id=session_id,
        user_id=user_id,
        app_name='warehouse_app'
    )

    if not existing_session:
        await session_service.create_session(
            app_name='warehouse_app',
            user_id=user_id,
            session_id=session_id
        )
    
    runner = Runner(
        session_service=session_service,
        agent=warehouse_agent,
        app_name='warehouse_app'
    )

    content = types.Content(
        role="user",
        parts=[
            types.Part(
                text=query
            )
        ]
    )
    
    final_answer = ''

    for event in runner.run(
        user_id=user_id,
        session_id=session_id,
        new_message=content,
        run_config=RunConfig(
            max_llm_calls=5
        )
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    final_answer += part.text

    return final_answer

In [ ]:
answer_1 = await run_warehouse_agent(
    query="What's the availability of B09X1LDMH6 in all warehouses?",
    session_id='session_2',
    user_id='user_2',
    session_service=session_service
)
print(answer_1)

Availability check completed for **B09X1LDMH6**:

- **Can be fulfilled completely:** Yes
- **Warehouses with full availability:**
  - **FR-LYO-01** — Lyon Regional Warehouse, Lyon, France: **28 available**
  - **DE-MUN-01** — Munich Logistics Hub, Munich, Germany: **28 available**
  - **FR-PAR-01** — Paris Central Depot, Paris, France: **65 available**
  - **FR-MAR-01** — Marseille Mediterranean Hub, Marseille, France: **7 available**
  - **DE-HAM-01** — Hamburg North Warehouse, Hamburg, Germany: **60 available**
- **Warehouse with no availability:**
  - **DE-BER-01** — Berlin Distribution Center, Berlin, Germany: **0 available**

No partial availability issues were found.


In [ ]:
answer_2 = await run_warehouse_agent(
    query="Reserve 5 units of this in Marseille warehouse?",
    session_id='session_2',
    user_id='user_2',
    session_service=session_service
)
print(answer_2)

In [21]:
answer_3 = await run_warehouse_agent(
    query="What's the availability of B09X1LDMH6 in Marseille warehouse?",
    session_id='session_2',
    user_id='user_2',
    session_service=session_service
)
print(answer_3)

Availability for **B09X1LDMH6** in **Marseille Mediterranean Hub (FR-MAR-01)**:

- **Available:** 2 units

Actions performed:
- Checked warehouse availability for **B09X1LDMH6** in Marseille warehouse
